In [1]:
import pandas as pd

wine = pd.read_csv("https://raw.githubusercontent.com/rickiepark/hg-mldl/master/wine.csv")

In [2]:
data = wine[['alcohol','sugar','pH']].to_numpy()
target = wine["class"].to_numpy()

In [3]:
from sklearn.model_selection import train_test_split

train_input,test_input,train_target,test_target = train_test_split(data,target,test_size=0.2,random_state=42)

In [5]:
sub_input, val_input, sub_target, val_target = train_test_split(train_input,train_target,test_size=0.2,random_state=42)

In [6]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(sub_input,sub_target)
print(dt.score(sub_input,sub_target))
print(dt.score(val_input,val_target))

0.9971133028626413
0.8653846153846154


In [ ]:
from sklearn.model_selection import cross_validate
#교차 검증
score = cross_validate(dt,train_input,train_target)
score

{'fit_time': array([0.01206708, 0.01171732, 0.01233482, 0.01304078, 0.01107836]),
 'score_time': array([0.00251389, 0.00252509, 0.00200105, 0.00125408, 0.00199795]),
 'test_score': array([0.86346154, 0.84711538, 0.87584216, 0.85466795, 0.83830606])}

In [10]:
import numpy as np

print(np.mean(score['test_score']))

0.8558786184941141


In [11]:
from sklearn.model_selection import StratifiedKFold

scores= cross_validate(dt,train_input,train_target,cv=StratifiedKFold())
print(np.mean(scores["test_score"]))

0.8570317242911084


In [12]:
splitter = StratifiedKFold(n_splits=10,shuffle=True,random_state=42)
scores = cross_validate(dt,train_input,train_target,cv = splitter)
print(np.mean(scores["test_score"]))

0.8543397065362383


In [13]:
from sklearn.model_selection import GridSearchCV

params = {"min_impurity_decrease":[0.0001,0.0002,0.0003,0.0004,0.0005]}

gs = GridSearchCV(DecisionTreeClassifier(random_state=42),params,n_jobs=1)

gs.fit(train_input,train_target)

dt = gs.best_estimator_
print(dt.score(train_input,train_target))

0.9615162593804117


In [15]:
print(gs.best_params_)
print(gs.cv_results_["mean_test_score"])
best_index = np.argmax(gs.cv_results_["mean_test_score"])
print(gs.cv_results_["params"][best_index])

{'min_impurity_decrease': 0.0001}
[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]
{'min_impurity_decrease': 0.0001}


In [17]:
params = {
    "min_impurity_decrease" : np.arange(0.0001,0.001,0.0001),
    "max_depth" : range(5,20,1),
    "min_samples_split": range(2,100,10)
}

In [18]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42),params,n_jobs=1)
gs.fit(train_input,train_target)
print(gs.best_params_)

#GridSearch

{'max_depth': 14, 'min_impurity_decrease': 0.0004, 'min_samples_split': 12}


In [ ]:
from scipy.stats import uniform,randint

rgen = randint(0,10)
rgen.rvs(10)
np.unique(rgen.rvs(1000),return_counts=True)

array([4, 6, 5, 6, 6, 8, 3, 9, 3, 9], dtype=int64)

In [22]:
ugen = uniform(0,1)
ugen.rvs(10)

array([0.03391885, 0.79984096, 0.29837073, 0.1620837 , 0.99684562,
       0.9629952 , 0.64569362, 0.3934852 , 0.98609163, 0.92184058])

In [29]:
params = {
    "min_impurity_decrease" : uniform(0.0001,0.001),
    "max_depth" : randint(20,50),
    "min_samples_split": randint(2,25),
    "min_samples_leaf" : randint(1,25)
}

from sklearn.model_selection import RandomizedSearchCV

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42),params, n_jobs = 1,n_iter=100,random_state=42)
rs.fit(train_input,train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026CEDA13CE0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000026CEE78F3B0>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026CEE4E7C20>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026CEE4E77D0>},
                   random_state=42)

In [30]:
print(rs.best_params_)
print(np.max(rs.cv_results_["mean_test_score"]))

{'max_depth': 39, 'min_impurity_decrease': 0.00034102546602601173, 'min_samples_leaf': 7, 'min_samples_split': 13}
0.8695428296438884


In [31]:
dt = gs.best_estimator_
print(dt.score(test_input,test_target))

0.8615384615384616
